# Adjacency matrix

In [ ]:
from lingam import DirectLiNGAM

model = DirectLiNGAM()
model.fit(reduced_data)

# Extract adjacency matrix
adj_matrix = model.adjacency_matrix_

In [ ]:
import networkx as nx

# Create a directed graph
causal_graph = nx.DiGraph()
num_features = reduced_data.shape[1]

# Add edges based on LiNGAM's adjacency matrix
for i in range(num_features):
  for j in range(num_features):
    if adj_matrix[i, j] != 0:  # Nonzero values indicate causal relationships
      causal_graph.add_edge(i, j, weight=adj_matrix[i, j])

In [ ]:
# Create a DataFrame for the reduced data with original column names
reduced_data_df = pd.DataFrame(data=reduced_data, index=norm_df.index, columns=[norm_df.columns[i] for i in range(100)])

# CausalNN

In [ ]:
# =============================
# Step 2: Define the CausalNN Model
# =============================
class CausalNN(Model):
    def __init__(self, input_dim, latent_dim, causal_graph):
        """
        CausalNN for dimensionality reduction, integrating a causal graph.

        :param input_dim: Number of input features.
        :param latent_dim: Size of the reduced representation.
        :param causal_graph: NetworkX graph defining causal feature relationships.
        """
        super(CausalNN, self).__init__()
        self.latent_dim = latent_dim
        self.causal_graph = causal_graph  # Store learned causal structure

        # Encoder
        self.encoder = tf.keras.Sequential([
            layers.InputLayer(input_shape=(input_dim,)),
            layers.Dense(128, activation='leaky_relu', activity_regularizer=tf.keras.regularizers.l1(1e-5)),
            layers.Dropout(0.2),  # Prevents redundancy
            layers.Dense(64, activation='leaky_relu'),
            layers.Dense(latent_dim, activation='tanh')  # Tanh captures non-linear interactions
        ])

        # Decoder
        self.decoder = tf.keras.Sequential([
            layers.InputLayer(input_shape=(latent_dim,)),
            layers.Dense(64, activation='leaky_relu'),
            layers.Dense(128, activation='leaky_relu'),
            layers.Dense(input_dim, activation=None)  # Matches z-score normalization
        ])

    def call(self, inputs):
        encoded = self.encoder(inputs)
        decoded = self.decoder(encoded)
        return decoded

    def compute_causal_penalty(self, inputs):
        """
        Computes a causal loss to enforce learned representations to respect the causal structure.
        """
        adjacency_matrix = nx.to_numpy_array(self.causal_graph)
        adjacency_matrix = tf.convert_to_tensor(adjacency_matrix, dtype=tf.float32)

        # Get the encoded feature representation
        encoded = self.encoder(inputs)

        # Select only nodes that exist in the causal graph
        encoded_subset = tf.gather(encoded, indices=list(self.causal_graph.nodes), axis=-1)

        # Reshape for proper matrix multiplication
        encoded_subset = tf.reshape(encoded_subset, [tf.shape(encoded_subset)[0], len(self.causal_graph.nodes)])

        # Enforce causal consistency by penalizing violations in causal structure
        causal_penalty = tf.reduce_sum(tf.abs(tf.matmul(encoded_subset, adjacency_matrix)))

        return causal_penalty

# =============================
# Step 3: Define the Loss Function
# =============================
def compute_loss(model, x, reconstructed):
    mse_loss = tf.keras.losses.MeanSquaredError()(x, reconstructed)
    mae_loss = tf.keras.losses.MeanAbsoluteError()(x, reconstructed)
    causal_loss = model.compute_causal_penalty(x) * 0.01  # Small weight for causal regularization
    return 0.5 * mse_loss + 0.5 * mae_loss + causal_loss

# =============================
# Step 4: Training Function
# =============================
def train_causal_nn(data, input_dim, latent_dim, causal_graph, epochs=50, batch_size=64):
    """
    Trains the CausalNN model.

    :param data: NumPy array of omics data.
    :param input_dim: Number of input features.
    :param latent_dim: Dimension of reduced feature space.
    :param causal_graph: Causal structure learned from LiNGAM.
    :param epochs: Number of training epochs.
    :param batch_size: Batch size for optimization.
    :return: Trained CausalNN model.
    """
    model = CausalNN(input_dim, latent_dim, causal_graph)
    optimizer = tf.keras.optimizers.Adam()

    for epoch in range(epochs):
        for batch_start in range(0, data.shape[0], batch_size):
            batch_data = data[batch_start:batch_start + batch_size]
            with tf.GradientTape() as tape:
                reconstructed = model(batch_data)
                loss = compute_loss(model, batch_data, reconstructed)
            gradients = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        print(f"Epoch {epoch + 1}, Loss: {loss.numpy():.4f}")

    return model

In [ ]:
# =============================
# Step 5: Run the Pipeline
# =============================
if __name__ == "__main__":
    import pandas as pd

    # Load omics dataset (assumed to be preprocessed)
    data_array = reduced_data_df.values.astype(np.float32)

    # Define model parameters
    input_dim = data_array.shape[1]
    latent_dim = 100
    epochs = 50
    batch_size = 64

    # Train the CausalNN model
    causal_nn = train_causal_nn(data_array, input_dim, latent_dim, causal_graph, epochs, batch_size)

    # Encode the data into the latent space
    reduced_shape = causal_nn.encoder(data_array).numpy()

    print("Reduced data shape:", reduced_shape.shape)

In [ ]:
model1 = DirectLiNGAM()
model1.fit(reduced_data1)

# Extract adjacency matrix
adj_matrix1 = model1.adjacency_matrix_

In [ ]:
# Create a directed graph
causal_graph1 = nx.DiGraph()
num_features = reduced_data1.shape[1]

# Add edges based on LiNGAM's adjacency matrix
for i in range(num_features):
  for j in range(num_features):
    if adj_matrix1[i, j] != 0:  # Nonzero values indicate causal relationships
      causal_graph1.add_edge(i, j, weight=adj_matrix1[i, j])

In [ ]:
# Create a DataFrame for the reduced data with original column names
reduced_data_df1 = pd.DataFrame(data=reduced_data1, index=norm_df1.index, columns=[norm_df1.columns[i] for i in range(100)])

In [ ]:
# =============================
# Step 5: Run the Pipeline
# =============================
if __name__ == "__main__":
    import pandas as pd

    # Load omics dataset (assumed to be preprocessed)
    data_array1 = reduced_data_df1.values.astype(np.float32)

    # Define model parameters
    input_dim = data_array1.shape[1]
    latent_dim = 100
    epochs = 50
    batch_size = 64

    # Train the CausalNN model
    causal_nn = train_causal_nn(data_array1, input_dim, latent_dim, causal_graph1, epochs, batch_size)

    # Encode the data into the latent space
    reduced_shape1 = causal_nn.encoder(data_array1).numpy()

    print("Reduced data shape:", reduced_shape1.shape)

In [ ]:
model2 = DirectLiNGAM()
model2.fit(reduced_data2)

# Extract adjacency matrix
adj_matrix2 = model2.adjacency_matrix_

In [ ]:
# Create a directed graph
causal_graph2 = nx.DiGraph()
num_features = reduced_data2.shape[1]

# Add edges based on LiNGAM's adjacency matrix
for i in range(num_features):
  for j in range(num_features):
    if adj_matrix2[i, j] != 0:  # Nonzero values indicate causal relationships
      causal_graph2.add_edge(i, j, weight=adj_matrix2[i, j])

In [ ]:
# Create a DataFrame for the reduced data with original column names
reduced_data_df2 = pd.DataFrame(data=reduced_data2, index=norm_df2.index, columns=[norm_df2.columns[i] for i in range(50)])

In [ ]:
# =============================
# Step 5: Run the Pipeline
# =============================
if __name__ == "__main__":
    import pandas as pd

    # Load omics dataset (assumed to be preprocessed)
    data_array2 = reduced_data_df2.values.astype(np.float32)
    # Define model parameters
    input_dim = data_array2.shape[1]
    latent_dim = 50
    epochs = 50
    batch_size = 64

    # Train the CausalNN model
    causal_nn = train_causal_nn(data_array2, input_dim, latent_dim, causal_graph2, epochs, batch_size)

    # Encode the data into the latent space
    reduced_shape2 = causal_nn.encoder(data_array2).numpy()

    print("Reduced data shape:", reduced_shape2.shape)

In [ ]:
model3 = DirectLiNGAM()
model3.fit(reduced_data3)

# Extract adjacency matrix
adj_matrix3 = model3.adjacency_matrix_

In [ ]:
# Create a directed graph
causal_graph3 = nx.DiGraph()
num_features = reduced_data3.shape[1]

# Add edges based on LiNGAM's adjacency matrix
for i in range(num_features):
  for j in range(num_features):
    if adj_matrix3[i, j] != 0:  # Nonzero values indicate causal relationships
      causal_graph3.add_edge(i, j, weight=adj_matrix3[i, j])

In [ ]:
# Create a DataFrame for the reduced data with original column names
reduced_data_df3 = pd.DataFrame(data=reduced_data3, index=norm_df3.index, columns=[norm_df3.columns[i] for i in range(100)])

In [ ]:
# =============================
# Step 5: Run the Pipeline
# =============================
if __name__ == "__main__":
    import pandas as pd

    # Load omics dataset (assumed to be preprocessed)
    data_array3 = reduced_data_df3.values.astype(np.float32)

    # Define model parameters
    input_dim = data_array3.shape[1]
    latent_dim = 100
    epochs = 50
    batch_size = 64

    # Train the CausalNN model
    causal_nn = train_causal_nn(data_array3, input_dim, latent_dim, causal_graph3, epochs, batch_size)

    # Encode the data into the latent space
    reduced_shape3 = causal_nn.encoder(data_array3).numpy()

    print("Reduced data shape:", reduced_shape3.shape)